This is the version without player ID and movement 

if you run into an issue with the vir env try this: 

brandon@Mac ~ % cd /Users/brandon/Desktop/Golf_Data/
brandon@Mac Golf_Data % ls -al

In [16]:
import os
import pandas as pd
import re
import numpy as np
pd.set_option('display.max_rows', None)


''' To upload SG files do the following: 
1. go to pga.com site and go to stats site, then to the SG:Total 
2. Save the file as SG_[tourneyID]_[year].csv 
3. Save the file to the /Users/brandon/Desktop/Golf_Data/upload_staging/data_for_upload/ folder 

Other info: 
4. the process should create a merged file (this can be deleted at some point) in the folder 

'''

# Base folder path
# base_path = "/Users/brandon/Desktop/Golf_Data/upload_staging/sg_upload"
base_path = r"C:\Users\brand\OneDrive\Documents\pga_stats\upload_staging\sg_upload"


# File to join
#join_file = "/Users/brandon/Desktop/Golf_Data/PGA_CourseInfo_for_Upload.csv"
join_file = r"C:\Users\brand\OneDrive\Documents\pga_stats\PGA_CourseInfo_for_Upload.csv"

# Load the file to join
main_df = pd.read_csv(join_file)
main_df = main_df[['TourneyID', 'Tournament', 'Week_Of_Season', 'Course', 'Signature_Event', 'Major_Event', 'par']]

# --- Parse ID and Year from filename ---
def parse_id_and_year(filename):
    match = re.match(r'.*_(\d+)_(\d{4})\.csv', filename)
    if match:
        return match.group(1), match.group(2)
    return None, None


# --- Identify Leader Files ---
leader_files = [
    f for f in os.listdir(base_path)
    if f.startswith('sg') and f.endswith('.csv')
]

In [17]:
# --- Process Each Leader File ---
for leader_file in leader_files:
    tourney_id, year = parse_id_and_year(leader_file)
    if not tourney_id:
        continue

    print(f"Processing SG file: {leader_file} (TourneyID={tourney_id}, year={year})")

    # Load leader file
    leader_path = os.path.join(base_path, leader_file)
    leader_df = pd.read_csv(leader_path)
    leader_df['TourneyID'] = int(tourney_id)
    leader_df['year'] = year

    # Join metadata (left join on tourneyID)
    enriched_df = leader_df.merge(main_df, on='TourneyID', how='left')

    # Remove blank rows
    enriched_df.dropna(how='all', inplace=True)

    # Trim spaces from column names in join file
    enriched_df.columns = enriched_df.columns.str.strip()

    # Trim spaces from string values in join file
    enriched_df = enriched_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)



    # Save output


Processing SG file: sg_6_2026.csv (TourneyID=6, year=2026)


In [18]:
tourney_file_name = enriched_df['Tournament'].unique().tolist()
print(tourney_file_name)

['Cognizant Classic in The Palm Beaches']


In [19]:
# Need to re-do postgres creds 
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values

# Load environment variables from .env file
load_dotenv()

# Database configuration from environment variables
db_config = {
    'host': os.getenv('DB_HOST'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'port': os.getenv('DB_PORT', '5432')
}
schema = os.getenv('DB_SCHEMA')

# Test the connection
try:
    conn = psycopg2.connect(**db_config)
    conn.close()
    print("✅ Database connection successful!")
    print(f"   Connected to: {db_config['database']} on {db_config['host']}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
   Connected to: brdb on localhost


In [20]:
# 
# use this to insert a group of csv files into a table 
'''
import psycopg2
import pandas as pd
import os


conn = psycopg2.connect(
    dbname='brdb',
    user='brandon',
    password='',
    host='localhost',
    port='5432'
)

cur = conn.cursor()
'''
# Folder where your CSV files are stored
# csv_folder = '/Users/brandon/Desktop/Golf_Data/test/the_players_championshipSG_combined_Data.csv'



"\nimport psycopg2\nimport pandas as pd\nimport os\n\n\nconn = psycopg2.connect(\n    dbname='brdb',\n    user='brandon',\n    password='',\n    host='localhost',\n    port='5432'\n)\n\ncur = conn.cursor()\n"

In [21]:
# Target table name
table_name = 'sg_data'

In [22]:
#adding 2 null cols for Movement and PlayerID 
enriched_df['MOVEMENT'] = np.nan
#enriched_df['PLAYER_ID'] = np.nan
enriched_df['PLAYER_ID'] = None
#enriched_df['PLAYER_ID'] = pd.to_numeric(enriched_df['PLAYER_ID'], errors='coerce').astype('Int64')
#enriched_df = enriched_df.where(pd.notna(enriched_df), None)


#enriched_df['MOVEMENT'] = pd.Series([np.nan] * len(enriched_df), dtype='Int64')  # note the capital 'I'
#enriched_df['PLAYER_ID'] = pd.Series([np.nan] * len(enriched_df), dtype='Int64')  # note the capital 'I'


# reorder cols 
enriched_df = enriched_df[['Rank', 'MOVEMENT', 'PLAYER_ID', 'Player','Avg', 'Total SG:T', 'Total SG:T2G'
                           , 'Total SG:P', 'Measured Rounds','TourneyID','year','Tournament'
                           , 'Week_Of_Season' , 'Course','Signature_Event','Major_Event','par']]




In [23]:
enriched_df

,Rank,MOVEMENT,PLAYER_ID,Player,Avg,Total SG:T,Total SG:T2G,Total SG:P,Measured Rounds,TourneyID,year,Tournament,Week_Of_Season,Course,Signature_Event,Major_Event,par
0,1,NaN,None,Nico Echavarria,3.508,14.033,9.761,4.273,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
1,2,NaN,None,Shane Lowry,3.008,12.033,6.969,5.065,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
2,2,NaN,None,Taylor Moore,3.008,12.033,7.744,4.290,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
3,2,NaN,None,Austin Smotherman,3.008,12.033,9.474,2.560,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
4,5,NaN,None,Ricky Castillo,2.508,10.033,8.080,1.954,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
5,6,NaN,None,Nicolai Højgaard,2.008,8.033,7.002,1.032,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
6,6,NaN,None,Keith Mitchell,2.008,8.033,6.036,1.998,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
7,6,NaN,None,William Mouw,2.008,8.033,3.907,4.127,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
8,9,NaN,None,Joel Dahmen,1.758,7.033,7.656,-0.622,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
9,9,NaN,None,Rasmus Højgaard,1.758,7.033,4.124,2.910,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0


In [24]:
df = enriched_df.rename(columns={'Rank': 'Rank', 'MOVEMENT': 'Movement', 
                            'PLAYER_ID': 'PlayerID', 'Player': 'Player', 'Total SG:T': 'Total_SG_T'
                            , 'Total SG:T2G': 'Total_SG_T2G', 'Total SG:P':'Total_SG_P'
                            , 'Measured Rounds':'Measured_Rounds' })

df = df.dropna(subset=df.columns[:7], how='all')

columns_to_clean = ['par', 'Week_Of_Season', 'year','Rank','Measured_Rounds'] # removing Player_ID since its null/blank



#print(df.head())

In [25]:
df

,Rank,Movement,PlayerID,Player,Avg,Total_SG_T,Total_SG_T2G,Total_SG_P,Measured_Rounds,TourneyID,year,Tournament,Week_Of_Season,Course,Signature_Event,Major_Event,par
0,1,NaN,None,Nico Echavarria,3.508,14.033,9.761,4.273,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
1,2,NaN,None,Shane Lowry,3.008,12.033,6.969,5.065,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
2,2,NaN,None,Taylor Moore,3.008,12.033,7.744,4.290,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
3,2,NaN,None,Austin Smotherman,3.008,12.033,9.474,2.560,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
4,5,NaN,None,Ricky Castillo,2.508,10.033,8.080,1.954,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
5,6,NaN,None,Nicolai Højgaard,2.008,8.033,7.002,1.032,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
6,6,NaN,None,Keith Mitchell,2.008,8.033,6.036,1.998,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
7,6,NaN,None,William Mouw,2.008,8.033,3.907,4.127,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
8,9,NaN,None,Joel Dahmen,1.758,7.033,7.656,-0.622,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0
9,9,NaN,None,Rasmus Højgaard,1.758,7.033,4.124,2.910,4,6,2026,Cognizant Classic in The Palm Beaches,9.0,PGA National Resort,N,N,71.0


In [26]:
df.drop(columns=['TourneyID'], inplace=True)

In [27]:
for col in columns_to_clean:
    df[col] = df[col].apply(lambda x: None if pd.isna(x) else int(x))

In [28]:
#df = df.dropna(subset=['pos'])
#df = df.dropna(how='all')

In [29]:
df

,Rank,Movement,PlayerID,Player,Avg,Total_SG_T,Total_SG_T2G,Total_SG_P,Measured_Rounds,year,Tournament,Week_Of_Season,Course,Signature_Event,Major_Event,par
0,1,NaN,None,Nico Echavarria,3.508,14.033,9.761,4.273,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
1,2,NaN,None,Shane Lowry,3.008,12.033,6.969,5.065,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
2,2,NaN,None,Taylor Moore,3.008,12.033,7.744,4.290,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
3,2,NaN,None,Austin Smotherman,3.008,12.033,9.474,2.560,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
4,5,NaN,None,Ricky Castillo,2.508,10.033,8.080,1.954,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
5,6,NaN,None,Nicolai Højgaard,2.008,8.033,7.002,1.032,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
6,6,NaN,None,Keith Mitchell,2.008,8.033,6.036,1.998,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
7,6,NaN,None,William Mouw,2.008,8.033,3.907,4.127,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
8,9,NaN,None,Joel Dahmen,1.758,7.033,7.656,-0.622,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71
9,9,NaN,None,Rasmus Højgaard,1.758,7.033,4.124,2.910,4,2026,Cognizant Classic in The Palm Beaches,9,PGA National Resort,N,N,71


In [30]:
conn = psycopg2.connect(**db_config)
cur = conn.cursor()
for _, row in df.iterrows():
    columns = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(row))
    insert_sql = f"INSERT INTO {schema}.{table_name} ({columns}) VALUES ({placeholders})"
    
    # DEBUG: print the row and data types
    print("Row data:", row.to_dict())

    try:
        cur.execute(insert_sql, tuple(row))
    except Exception as e:
        print("Error on row:", row.to_dict())
        print("Data types:", [type(x) for x in row])
        raise e  # re-raise to stop execution

    #cur.execute(insert_sql, tuple(row))
        
    conn.commit()
    print(f"Inserted data from {tourney_file_name} into {table_name}")

Row data: {'Rank': 1, 'Movement': nan, 'PlayerID': None, 'Player': 'Nico Echavarria', 'Avg': 3.508, 'Total_SG_T': 14.033, 'Total_SG_T2G': 9.761, 'Total_SG_P': 4.273, 'Measured_Rounds': 4, 'year': 2026, 'Tournament': 'Cognizant Classic in The Palm Beaches', 'Week_Of_Season': 9, 'Course': 'PGA National Resort', 'Signature_Event': 'N', 'Major_Event': 'N', 'par': 71}
Inserted data from ['Cognizant Classic in The Palm Beaches'] into sg_data
Row data: {'Rank': 2, 'Movement': nan, 'PlayerID': None, 'Player': 'Shane Lowry', 'Avg': 3.008, 'Total_SG_T': 12.033, 'Total_SG_T2G': 6.969, 'Total_SG_P': 5.065, 'Measured_Rounds': 4, 'year': 2026, 'Tournament': 'Cognizant Classic in The Palm Beaches', 'Week_Of_Season': 9, 'Course': 'PGA National Resort', 'Signature_Event': 'N', 'Major_Event': 'N', 'par': 71}
Inserted data from ['Cognizant Classic in The Palm Beaches'] into sg_data
Row data: {'Rank': 2, 'Movement': nan, 'PlayerID': None, 'Player': 'Taylor Moore', 'Avg': 3.008, 'Total_SG_T': 12.033, 'Tota